[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Reading What They Learned


## What you will be able to do

Ask a LightGBM or XGBoost model which features it used, by counting its splits, by adding up their
gain, and by splitting each prediction into a contribution from every feature, and know what each of
those numbers can and cannot tell you.

## The idea

### The problem

A model of 200 trees that predicts a night's lowest temperature to within a degree or two is useful,
and a forecaster will still ask why it predicts −19 degrees for one station and −3 for another. A
linear model answers with its weights. A sum of 200 trees with up to 31 leaves each has thousands of
splits and no single number for any feature.

A forecaster can ask which features the model relies on across all the stations, and, for one
particular night, which features pushed the prediction down and by how much. The libraries answer the first with importances, and both questions with contributions.

### A formal definition

> A **feature importance** is a number per feature that summarizes how the trees used it, by the
> number of splits made on it (**split** in LightGBM, **weight** in XGBoost) or by how much those
> splits reduced the training loss (**gain**, a total in LightGBM and an average per split in XGBoost).
> A **SHAP contribution** is an amount per feature for one prediction. The contributions of all the
> features plus a bias term, the same for every row, add up exactly to the model's raw output for that
> row, which is the predicted value for a regression and the log-odds for a classifier.

### Why it works that way

Counting splits is cheap and easy to misread. A feature with many distinct values, such as the wind
speed, offers a split at almost any point, and the trees use it often for small refinements, while
a feature that matters a great deal can settle its question in a few splits. A column of pure noise
also offers many split points, so it collects splits that reduce the loss by almost nothing. Gain
weighs each split by what it achieved, which is closer to "how much the model relies on this".

Contributions come from Shapley values, a way of sharing a total among players by averaging each
player's effect over every order in which the players could join. For trees there is an exact way to
compute them, TreeSHAP, and both libraries have it built in. Their sum is the prediction, so they
explain one night in degrees, and their average size across many nights is an importance in degrees
too. Because each contribution depends on the other features' values, the same clear sky can count
for more on a winter night than on a summer one, which no single importance number can show.

### Where this shows up

The **scikit-learn, Deep Dive** guide measured a forest's importances with permutation, and the
**A Frost Forecast** notebook reads its classifier the same way as this one. The `shap` package, which
Colab also has, computes the same kind of contribution with its `TreeExplainer` and draws it as plots.

### What this notebook covers

- importance by split counts and by gain in LightGBM, and XGBoost's five importance types
- contributions for every prediction in both libraries, and the check that they add up
- contributions averaged into an importance, and one cold night explained
- how a clear sky's contribution changes with the season
- four mistakes in reading what a model learned

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import lightgbm as lgb
import pandas as pd

x = pd.DataFrame({"a": [0, 0, 1, 1] * 25, "b": [0, 1, 0, 1] * 25})
y = 3 * x["a"] + x["b"]
model = lgb.LGBMRegressor(n_estimators=100, min_child_samples=5, verbose=-1).fit(x, y)
print(model.predict(x[:4]).round(2))
print(model.predict(x[:4], pred_contrib=True).round(2))
```

```
[0. 1. 3. 4.]
[[-1.5 -0.5  2. ]
 [-1.5  0.5  2. ]
 [ 1.5 -0.5  2. ]
 [ 1.5  0.5  2. ]]
```

The answer is 3 × `a` + `b`, and the model learned it. With `pred_contrib=True`, each row splits its
prediction into a contribution from `a`, one from `b`, and the average prediction, 2, in the last
column. For the fourth row, 1.5 + 0.5 + 2 = 4. `a` moves a prediction three times as far as `b` does.

## Setup

Two libraries for the table, the two boosting libraries, and one score.

- `numpy` and `pandas` build the table and the tables of importances
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `mean_squared_error` checks that the two models are worth reading

`station_days` makes the same 20,000 made-up station-days as the rest of the guide, from a fixed
seed, so the table is the same on every run, and the cell splits the stations into 300 for training,
50 for validation and 50 for testing. The models train on the training stations, and their
contributions are computed for the test stations.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 test rows


## Worked examples

### The models to read

A LightGBM and an XGBoost model of the day's lowest temperature, each with 200 trees at a learning
rate of 0.1.

In [2]:
lightgbm_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
lightgbm_model.fit(train[FEATURES], train["tmin"])
xgboost_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=0)
xgboost_model.fit(train[FEATURES], train["tmin"])

for name, model in (("LightGBM", lightgbm_model), ("XGBoost", xgboost_model)):
    print(name, "test loss", round(mean_squared_error(test["tmin"], model.predict(test[FEATURES])), 3))

LightGBM test loss 2.552
XGBoost test loss 2.621


The two models score 2.552 and 2.621 on the test stations, close to the guide's earlier boosted
models, so what they learned is worth reading.

### Importance by splits and by gain in LightGBM

`feature_importances_` follows the model's `importance_type`, which for LightGBM is `"split"`, the
number of splits made on each feature. The booster's `feature_importance("gain")` adds up how much
each feature's splits reduced the loss.

In [3]:
print("importance_type:", lightgbm_model.importance_type)
lightgbm_importance = pd.DataFrame({
    "splits": lightgbm_model.feature_importances_,
    "gain": lightgbm_model.booster_.feature_importance("gain").round(0).astype(int),
}, index=lightgbm_model.feature_name_)
lightgbm_importance

importance_type: split


,splits,gain
latitude,961,353164
elevation_m,889,102352
coast_km,757,38385
terrain,226,23658
day_of_year,1190,2308021
cloud_cover,784,91972
wind_ms,751,66021
snow_cm,442,381975


By splits, `day_of_year` comes first with 1,190 and `latitude` second with 961, and `snow_cm` has only
442, ahead of `terrain` alone. By gain the order changes. `day_of_year` is still far ahead, at
2,308,021, but `snow_cm` is second, at 381,975, just above `latitude`, and `elevation_m`, with twice
the splits of `snow_cm`, has about a quarter of its gain. Each of `snow_cm`'s splits reduced the loss
more than seven times as much as one of `elevation_m`'s, on average.

### XGBoost's five kinds of importance

XGBoost's booster reports five types through `get_score`. `weight` counts splits, `gain` is the
average gain of a feature's splits and `total_gain` their sum, and `cover` is the average number of
training rows a feature's splits sorted, with `total_cover` their sum. XGBoost counts each row by the
second derivative of the loss, which is 1 for a squared error, so these are row counts here.

In [4]:
booster = xgboost_model.get_booster()
xgboost_importance = pd.DataFrame({
    kind: booster.get_score(importance_type=kind) for kind in ("weight", "gain", "total_gain", "cover", "total_cover")
})
xgboost_importance.round(1)

,weight,gain,total_gain,cover,total_cover
latitude,1603.0,219.6,352003.3,2271.1,3640535.0
elevation_m,1378.0,74.9,103197.3,2210.9,3046654.0
coast_km,1101.0,35.3,38875.1,1777.0,1956431.0
terrain,648.0,45.7,29604.6,1166.1,755644.0
day_of_year,2039.0,1049.2,2139234.8,1705.0,3476426.0
cloud_cover,1617.0,57.5,93022.4,1024.2,1656159.0
wind_ms,1687.0,39.0,65782.3,1345.0,2269050.0
snow_cm,816.0,681.3,555966.0,1466.3,1196527.0


`weight` puts `day_of_year` first and then `wind_ms` and `cloud_cover`, while `total_gain` puts
`day_of_year`, `snow_cm` and `latitude` first, the same three as LightGBM's gain. `gain` is an average
per split, so `snow_cm`'s 816 splits average 681.3 each, against 39.0 for `wind_ms`'s 1,687. `cover`
is the average number of training rows one of a feature's splits sorted, and a split near the top of a
tree sorts more rows than one near a leaf.

In [5]:
gain = xgboost_importance["gain"]
print("feature_importances_ is gain divided by its total:",
      np.allclose(xgboost_model.feature_importances_, (gain / gain.sum()).to_numpy()))

feature_importances_ is gain divided by its total: True


XGBoost's `feature_importances_` is the average gain, divided by its total so that it sums to 1, while
LightGBM's counts splits. The two libraries' default importances answer different questions under
the same name, so a comparison of the two needs the type named.

### Contributions for every prediction

LightGBM's `predict` takes `pred_contrib=True` and returns one column per feature and a last column
for the average prediction. XGBoost computes the same thing through its booster, whose `predict` takes
a `DMatrix` and `pred_contribs=True`. XGBoost 3.4 reads `terrain` as a category without being asked, as
**The Native Interfaces** notebook showed, and the cell passes `enable_categorical=True` as older code
does.

In [6]:
lightgbm_contributions = lightgbm_model.predict(test[FEATURES], pred_contrib=True)
test_matrix = xgb.DMatrix(test[FEATURES], enable_categorical=True)
xgboost_contributions = booster.predict(test_matrix, pred_contribs=True)

print("shapes:", lightgbm_contributions.shape, xgboost_contributions.shape)
print("LightGBM rows add up to its predictions:",
      np.allclose(lightgbm_contributions.sum(axis=1), lightgbm_model.predict(test[FEATURES])))
print("XGBoost rows add up to its predictions:",
      np.allclose(xgboost_contributions.sum(axis=1), xgboost_model.predict(test[FEATURES]), atol=1e-4))
print("average prediction column:", round(float(lightgbm_contributions[0, -1]), 3),
      round(float(xgboost_contributions[0, -1]), 3), "| training mean:", round(float(train["tmin"].mean()), 3))
print("the same in every row:", np.allclose(lightgbm_contributions[:, -1], lightgbm_contributions[0, -1]),
      np.allclose(xgboost_contributions[:, -1], xgboost_contributions[0, -1]))

shapes: (2500, 9) (2500, 9)
LightGBM rows add up to its predictions: True
XGBoost rows add up to its predictions: True
average prediction column: -1.522 -1.523 | training mean: -1.522
the same in every row: True True


Both libraries give 9 numbers for each of the 2,500 test days, one per feature and the average
prediction, and every row adds up to its prediction. The last column is the same in every row,
−1.522 for LightGBM and −1.523 for XGBoost, and the cell prints the training days' mean lowest
temperature as −1.522. Both columns come from that mean, and XGBoost keeps its numbers to 32 bits, so
its last printed digit can round the other way.

### Contributions averaged into an importance

The average size of a feature's contribution over the 2,500 test days is an importance in degrees,
how far that feature moves a prediction on a typical day.

In [7]:
in_degrees = pd.DataFrame({
    "LightGBM": np.abs(lightgbm_contributions[:, :-1]).mean(axis=0),
    "XGBoost": np.abs(xgboost_contributions[:, :-1]).mean(axis=0),
}, index=FEATURES)
in_degrees.sort_values(["LightGBM", "XGBoost"], ascending=False, kind="stable").round(3)

,LightGBM,XGBoost
day_of_year,4.685,4.468
latitude,1.967,1.958
snow_cm,0.792,1.095
cloud_cover,0.783,0.780
elevation_m,0.686,0.665
wind_ms,0.584,0.579
coast_km,0.270,0.266
terrain,0.231,0.243


`day_of_year` moves a prediction about 4.7 degrees on a typical test day, more than the next two
features together. The two libraries rank the features in the same order, and differ most on
`snow_cm`, which XGBoost leans on more, 1.095 degrees against 0.792. `terrain` comes last in both, at
about 0.24 degrees.

### The coldest night explained

The test day the LightGBM model predicts coldest, its features, and each feature's contribution in
degrees, from the most negative.

In [8]:
coldest = int(np.argmin(lightgbm_model.predict(test[FEATURES])))
night = test.iloc[[coldest]]
print(night[FEATURES + ["tmin"]].to_string(index=False))

parts = pd.Series(lightgbm_contributions[coldest], index=FEATURES + ["average prediction"])
print(parts.sort_values(kind="stable").round(2).to_string())
print("sum:", round(float(parts.sum()), 2))

 latitude  elevation_m  coast_km terrain  day_of_year  cloud_cover  wind_ms  snow_cm  tmin
     64.5         1025      32.8   fjord           12         0.01      0.7     48.0 -19.0
day_of_year          -8.35
cloud_cover          -3.12
elevation_m          -2.93
wind_ms              -2.40
average prediction   -1.52
snow_cm              -1.18
latitude             -0.11
coast_km              0.00
terrain               0.16
sum: -19.47


The coldest prediction is −19.47 degrees, for a fjord station 1,025 meters up on 12 January, under a
clear sky with almost no wind and 48 centimeters of snow, where the day's lowest temperature was −19.0.
The season contributes −8.35 degrees, the clear sky −3.12, the height −2.93, the calm −2.40 and the snow
−1.18, and with the small contributions of the latitude, the coast and the terrain and the average
prediction, they add up to −19.47.

### A clear sky in winter and in summer

`cloud_cover`'s contribution, averaged over test days grouped by the sky and the season. The season
is the cosine used in the **Why Gradient Boosting** notebook, above 0.5 in the depth of winter and
below −0.5 in summer.

In [9]:
season = np.cos(2 * np.pi * (test["day_of_year"] - 20) / 365)
when = np.where(season > 0.5, "winter", np.where(season < -0.5, "summer", "spring or autumn"))
sky = pd.cut(test["cloud_cover"], [-0.01, 0.2, 0.8, 1.0], labels=["clear", "partly cloudy", "overcast"])
cloud = pd.Series(lightgbm_contributions[:, FEATURES.index("cloud_cover")], index=test.index)
cloud.groupby([when, sky], observed=True).agg(["mean", "size"]).round(2)

mean  size
                 cloud_cover              
spring or autumn clear         -1.29   115
                 partly cloudy -0.11   508
                 overcast       1.09   205
summer           clear         -1.01   123
                 partly cloudy -0.06   502
                 overcast       0.81   187
winter           clear         -1.84   153
                 partly cloudy -0.15   526
                 overcast       1.46   181

On clear winter days the sky lowers the prediction by 1.84 degrees on average, and on clear summer
days by 1.01, and an overcast sky raises it by 1.46 in winter and 0.81 in summer. The same cloud cover counts
for more in winter, the interaction between cloud and season that these made-up stations were built
with, and a single importance number for `cloud_cover` has no way to show it.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/13-reading-what-they-learned-solutions.ipynb).

**1.** Print LightGBM's importances by gain as a share of their total, sorted from the largest, beside its split counts as a share of theirs.

In [10]:
# your code here

**2.** Rank the features by XGBoost's `weight` and by its `total_gain`, and print the two rankings side by side.

In [11]:
# your code here

**3.** Find the test day the XGBoost model predicts warmest, and print each feature's contribution to that prediction from XGBoost's booster.

In [12]:
# your code here

**4.** Average `wind_ms`'s LightGBM contribution over test days grouped by the worked examples' `sky` groups and by whether the wind is below 2 meters a second, and say what the difference means.

In [13]:
# your code here

**5.** Train an `XGBClassifier` for frost (`tmin < 0`) with 200 trees and `random_state=0` on the training stations, take the contributions of the first test day from its booster, and turn their sum into a probability with `1 / (1 + np.exp(-total))` to compare with `predict_proba`.

In [14]:
# your code here

## Common errors

### No error, and a column of noise counted as more useful than snow

In [15]:
rng = np.random.default_rng(1)
with_noise = train[FEATURES].assign(noise=rng.normal(size=len(train)))
noisy = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, verbose=-1)
noisy.fit(with_noise, train["tmin"])
pd.DataFrame({
    "splits": noisy.feature_importances_,
    "gain": noisy.booster_.feature_importance("gain").round(0).astype(int),
}, index=noisy.feature_name_)

,splits,gain
latitude,894,352436
elevation_m,876,103333
coast_km,713,37811
terrain,216,23575
day_of_year,1052,2307385
cloud_cover,696,91444
wind_ms,662,65484
snow_cm,393,381655
noise,498,3712


A column of random numbers, unrelated to the temperature, gets 498 splits, more than `snow_cm`'s 393
or `terrain`'s 216. LightGBM sorts every numeric column into at most 255 bins, so the
noise offers about as many thresholds as `latitude` or `day_of_year`, far more than `snow_cm` or
`terrain`, and each split it makes fits a little of the noise in the training rows. Its gain, 3,712, is the smallest in the table, under a
sixth of `terrain`'s. When the question is what the model relies on, rank features by gain or by
contributions.

### `TypeError: XGBModel.predict() got an unexpected keyword argument 'pred_contribs'`

In [16]:
xgboost_model.predict(test[FEATURES], pred_contribs=True)

TypeError: XGBModel.predict() got an unexpected keyword argument 'pred_contribs'

The scikit-learn class's `predict` takes the data and a few options, and contributions are not among
them in XGBoost. They come from the booster, `xgboost_model.get_booster().predict(...)`, given a
`DMatrix`, as the worked examples do. LightGBM's `predict` does take `pred_contrib=True`.

### `KeyError: 'constant'`

In [17]:
with_constant = train[FEATURES].assign(constant=1.0)
constant_model = xgb.XGBRegressor(n_estimators=50, random_state=0).fit(with_constant, train["tmin"])
scores = constant_model.get_booster().get_score(importance_type="gain")
print(sorted(scores))
scores["constant"]

['cloud_cover', 'coast_km', 'day_of_year', 'elevation_m', 'latitude', 'snow_cm', 'terrain', 'wind_ms']


KeyError: 'constant'

`get_score` leaves out any feature the trees never split on, and a column with one value offers no
split, so `constant` is missing from the dictionary. Ask for a name with a default, or use
`feature_importances_`, which has an entry for every feature.

In [18]:
print("with a default:", scores.get("constant", 0.0))
print("feature_importances_:", round(float(constant_model.feature_importances_[list(with_constant.columns).index("constant")]), 3))

with a default: 0.0
feature_importances_: 0.0


### No error, and contributions that do not add up to the probability

In [19]:
frost = (train["tmin"] < 0).astype(int)
frost_model = lgb.LGBMClassifier(n_estimators=200, random_state=0, verbose=-1).fit(train[FEATURES], frost)
first_day = test[FEATURES].iloc[[0]]
total = float(frost_model.predict(first_day, pred_contrib=True)[0].sum())
print("sum of contributions:", round(total, 3))
print("probability of frost:", round(float(frost_model.predict_proba(first_day)[0, 1]), 3))

sum of contributions: -1.779
probability of frost: 0.144


For a classifier, LightGBM's contributions explain the raw score, in log-odds, and here they add up to
−1.779. The probability is that score passed through the logistic function, 1 / (1 + exp(1.779)), which
is 0.144. A contribution is a push in log-odds, and the same push moves the probability further near
one half than near 0 or 1. Turn the sum into a probability before comparing it with `predict_proba`.

## Recap

- You can read LightGBM's importances by split count and by gain, and XGBoost's five importance types.
- You can compute a contribution from every feature to every prediction in both libraries, and check
  that the contributions add up to the prediction.
- You can average contributions into an importance in the target's own units, and explain one
  prediction feature by feature.
- You can see an interaction, such as a clear sky that counts for more in winter, in contributions
  grouped by another feature.

## What is next

The **Saving and Loading Boosters** notebook writes a trained model to a file and reads it back, in
each library's formats.

---

&#8592; **Previous:** [Tuning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/12-tuning.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Saving and Loading Boosters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/14-saving-and-loading-boosters.ipynb) &#8594;
